### Retrievers in LangChain

A retriever is the part of a LangChain pipeline that finds the most relevant pieces of information from a document store before the LLM answers a question.

### What it does

When a user asks a question, the retriever:

1. Turns the question into an embedding.
2. Compares it with stored document embeddings.
3. Returns the most similar documents.
4. Passes those documents to a chain or model for final reasoning.

This is useful for RAG (Retrieval-Augmented Generation): the model does not rely only on its training memory, but also uses your own knowledge base.

### Why retrievers matter

Retrievers help with:

- knowledge-base Q&A
- private document search
- product documentation assistants
- customer support bots
- long-context retrieval without stuffing everything into the prompt

The key idea is simple: retrieve the right context, then ask the model to answer using that context.

### Typical flow

```python
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.load_local("my_chroma_db", embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

results = retriever.invoke("What is the company policy on remote work?")
for doc in results:
    print(doc.page_content)
```

In this flow:

- the query is embedded
- the vector store finds similar chunks
- the retrieved documents are passed to the model

### Types of retrievers

Common retrievers in LangChain include:

- VectorStoreRetriever: retrieves semantically similar chunks from a vector DB
- BM25Retriever: good for keyword-based retrieval
- ParentDocumentRetriever: keeps chunk-level matching while preserving original document context
- MultiQueryRetriever: generates several search queries to improve recall

### Best practices

- Split large documents into meaningful chunks before storing them.
- Use embeddings that match your domain and data type.
- Keep chunk size small enough for relevance, but large enough for context.
- Tune the number of retrieved documents (`k`) based on quality and token budget.
- Always validate that the retrieved context actually answers the question.

### Common mistakes

- Retrieving too many docs, which increases token cost and noise.
- Using very large chunks that bury the relevant passage.
- Forgetting to preprocess and split documents properly.
- Assuming a retriever is magic; it still needs good embeddings, data quality, and prompt design.


### Vector Store Retriever

A `VectorStoreRetriever` wraps a vector database and retrieves the documents whose embeddings are most similar to a user query.

Why this matters:
- It is the default pattern for semantic search in RAG systems.
- The query is embedded, the documents are embedded, and the store ranks them by similarity.
- This works well when the "meaning" of the question matters more than exact keyword matches.

When to use it:
- product docs and FAQ search
- private knowledge bases
- semantic document retrieval inside a chatbot

In [3]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
query = "What is Chroma used for?"
results = retriever.invoke(query)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

c:\Users\sidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [4]:
results = vectorstore.similarity_search(query, k=2)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


### Wikipedia Retriever

The `WikipediaRetriever` is a specialized retriever that pulls content directly from Wikipedia pages based on a query.

What is happening:
- It searches Wikipedia for relevant articles.
- It extracts the most relevant page content and returns it as LangChain `Document` objects.
- Those results can be fed into a prompt or a chain for answer generation.

Why use it:
- It is a fast way to ground a model with public, general knowledge.
- It is useful for experiments, demos, and quick information lookup.

When to use it:
- open-domain Q&A
- knowledge grounding on public topics
- quick prototype testing before building your own document retriever

In [2]:
from langchain_community.retrievers import WikipediaRetriever

# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display
## Vector Store Retriever

C:\Users\sidha\AppData\Local\Temp\ipykernel_17216\432049170.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever



--- Result 1 ---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,

### MMR (Maximal Marginal Relevance)

`MMR` is a retrieval strategy designed to reduce redundancy.

Instead of returning several nearly identical passages, MMR balances:
- similarity to the query
- diversity among returned documents

Why this matters:
- If the top results are too similar, the model may read repeated context and miss useful variety.
- MMR helps the retriever surface complementary passages that cover different angles of the same topic.

When to use it:
- broad research questions
- multi-aspect queries
- situations where repeated results would be low value

Common mistake:
- setting the diversity parameter too aggressively, which can make the results less relevant. Tuning `lambda_mult` is usually necessary.

Here, `search_type="mmr"` tells the retriever to aim for both relevance and diversity.

In [ ]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

query = "What is langchain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


### MultiQueryRetriever

The `MultiQueryRetriever` does not rely on a single search query. Instead, it asks an LLM to generate several alternative versions of the same question, then retrieves results for each one.

Why this helps:
- A user question may be phrased in ways that miss relevant documents.
- Generating multiple queries can improve recall when terms differ across sources.
- This is especially useful for ambiguous or broad questions.

When to use it:
- uncertain or underspecified user queries
- documents that use different terminology than the user
- search-heavy RAG pipelines with moderate complexity

Common mistake:
- generating too many variants can increase latency and retrieval noise. Keep the query set focused and relevant.

In practice, this retriever often performs better than a single-query search when the original phrasing is vague or semantically broad.

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

# Create retrievers
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
)

# Query
query = "How to improve energy levels and maintain balance?"

# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*" * 150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

--- Result 1 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and 

### Contextual Compression Retriever

A contextual compression retriever takes the initial search results and asks an LLM to compress or filter them before returning them to the final pipeline.

What is happening:
- The retriever first fetches a broader set of candidate documents.
- A compressor removes irrelevant, repetitive, or low-signal content.
- The final result is a cleaner context block for the model.

Why this matters:
- It reduces token usage.
- It improves answer quality by reducing noisy context.
- It works well when the initial retrieval is broad but not precise.

When to use it:
- long documents with lots of surrounding text
- noisy corpora
- RAG pipelines where context window and relevance matter

Common mistake:
- over-compressing too aggressively, which can remove needed details and damage accuracy. The goal is to keep the useful signal, not strip the passage down to nothing.

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Recreate the document objects from the previous data
docs = [
    Document(
        page_content=(
            """The Grand Canyon is one of the most visited natural wonders in the world.
            Photosynthesis is the process by which green plants convert sunlight into energy.
            Millions of tourists travel to see it every year. The rocks date back millions of years."""
        ),
        metadata={"source": "Doc1"}
    ),
    Document(
        page_content=(
            """In medieval Europe, castles were built primarily for defense.
            The chlorophyll in plant cells captures sunlight during photosynthesis.
            Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
        ),
        metadata={"source": "Doc2"}
    ),
    Document(
        page_content=(
            """Basketball was invented by Dr. James Naismith in the late 19th century.
            It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
        ),
        metadata={"source": "Doc3"}
    ),
    Document(
        page_content=(
            """The history of cinema began in the late 1800s. Silent films were the earliest form.
            Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
            Modern filmmaking involves complex CGI and sound design."""
        ),
        metadata={"source": "Doc4"}
    )
]

# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embedding_model)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
